In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score,accuracy_score, precision_score,roc_auc_score,f1_score,confusion_matrix
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')


## 0.파일 불러오기

In [33]:
df = pd.read_csv('../data/dataset/코스피_전처리완.csv')
X = df[df.columns[6:]]
y = df['분식기업']

## 1.데이터 split

In [34]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state = 42)

In [35]:
n, d = X_train.shape
print("number of feature:", d)  # 변수 개수
print("number of data:", n)     # 데이터 개수

number of feature: 40
number of data: 4916


In [36]:
X_train.describe()

,매출총이익률(%),영업이익률(%),당기순이익률(%),자본금영업이익률(%),영업수익/영업비용(%),ROE(세전계속사업이익)(%),자본금세전계속사업이익률(%),자본금지배주주순이익률(%),매출액증가율(전년동기)(%),영업이익증가율(전년동기)(%),...,현금흐름/총자본(%),영업현금흐름/투자현금흐름(%),DSRI,GMI,AQI,DEPI,SGAI,LVGI,TATA,벤포드
count,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.00000,4916.000000,4916.000000,4916.000000,...,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000
mean,22.034933,3.544138,1.702067,301.462667,104.943151,3.329184,314.71438,225.056932,18.775210,0.168910,...,8.600004,-0.944172,1.085161,1.074941,1.155167,0.037510,1.029087,1.018318,-0.010492,0.097030
std,20.453247,11.916821,36.035633,1359.432268,12.433844,47.948805,1614.55253,1716.880618,584.671105,2984.140321,...,37.323175,28.549077,1.676511,5.074056,1.424999,0.089438,0.470588,0.341328,0.062782,0.296029
min,-139.970000,-171.390000,-1148.680000,-2111.060000,36.850000,-2309.940000,-3145.61000,-3248.630000,-98.010000,-137079.350000,...,-1586.700000,-412.220000,0.000000,-150.020000,0.090000,0.000000,-22.520000,0.110000,-0.770000,0.000000
25%,10.230000,1.460000,0.200000,15.220000,101.487500,1.137500,6.11500,2.062500,-4.412500,-44.932500,...,5.060000,-1.780000,0.870000,0.900000,0.900000,0.000000,0.930000,0.930000,-0.040000,0.000000
50%,16.260000,4.120000,2.995000,82.940000,104.300000,6.900000,75.58500,52.900000,3.670000,-2.885000,...,10.550000,-0.860000,0.980000,1.000000,1.000000,0.010000,1.010000,0.990000,-0.010000,0.000000
75%,27.190000,7.490000,6.470000,252.600000,108.100000,13.012500,263.24500,186.040000,13.712500,37.650000,...,17.012500,-0.080000,1.100000,1.100000,1.130000,0.030000,1.090000,1.050000,0.020000,0.000000
max,100.000000,81.180000,1079.840000,50556.070000,531.400000,498.660000,49541.57000,100051.640000,40693.190000,116066.340000,...,228.330000,1256.760000,83.470000,185.240000,46.180000,0.980000,14.140000,10.180000,0.460000,1.000000


In [37]:
print(y_train.value_counts())
print(y_test.value_counts())

분식기업
0.0    4817
1.0      99
Name: count, dtype: int64
분식기업
0.0    2064
1.0      43
Name: count, dtype: int64


## 2. 모델링

In [38]:
model = LogisticRegression(random_state = 42)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.9786426198386331
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


In [39]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.9800650935720098
Precision: 0.6666666666666666
Recall: 0.020202020202020204
F1 Score: 0.03921568627450981


## 3. 이상치 처리

### 3-1. 상위 2%, 하위 2%, 상하위 1%씩

In [40]:
import numpy as np
from scipy.stats import mstats
import statsmodels.api as sm

# 중앙값과 절대 중앙값 편차 (MAD) 계산 함수
def mad_based_outliers(df, threshold=3):        # 임계값은 3으로 설정
    median = np.median(df)
    mad = sm.robust.scale.mad(df)
    mad_scaled = np.abs(df - median) / mad if mad != 0 else 0
    return mad_scaled > threshold

# 윈저라이징 조건 설정 함수
def apply_winsorization_based_on_outliers(df, threshold=3):
    winsorization_info = {}  # 각 컬럼별 윈저라이징 방식을 저장할 딕셔너리

    for column in df.columns:
        # MAD 기반 이상치 탐지
        outliers_mask = mad_based_outliers(df[column], threshold)
        outliers_high = np.sum(outliers_mask & (df[column] > np.median(df[column])))
        outliers_low = np.sum(outliers_mask & (df[column] < np.median(df[column])))

        # 조건에 따라 윈저라이징 방식 결정
        if outliers_high > 0 and outliers_low == 0:  # 상위 이상치만 있을 때
            df[column] = mstats.winsorize(df[column], limits=(0, 0.02))  # 상위 2%만 윈저라이징
            winsorization_info[column] = "상위 2% 윈저라이징"
        elif outliers_low > 0 and outliers_high == 0:  # 하위 이상치만 있을 때
            df[column] = mstats.winsorize(df[column], limits=(0.02, 0))  # 하위 2%만 윈저라이징
            winsorization_info[column] = "하위 2% 윈저라이징"
        elif outliers_high > 0 and outliers_low > 0:  # 상하위 이상치가 모두 있을 때
            df[column] = mstats.winsorize(df[column], limits=(0.01, 0.01))  # 상하위 1%씩 윈저라이징
            winsorization_info[column] = "상하위 1% 윈저라이징"
    return winsorization_info

In [41]:
winsorization_info = apply_winsorization_based_on_outliers(X_train)

In [42]:
X_train.describe()

,매출총이익률(%),영업이익률(%),당기순이익률(%),자본금영업이익률(%),영업수익/영업비용(%),ROE(세전계속사업이익)(%),자본금세전계속사업이익률(%),자본금지배주주순이익률(%),매출액증가율(전년동기)(%),영업이익증가율(전년동기)(%),...,현금흐름/총자본(%),영업현금흐름/투자현금흐름(%),DSRI,GMI,AQI,DEPI,SGAI,LVGI,TATA,벤포드
count,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.00000,4916.000000,4916.000000,4916.000000,...,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000,4916.000000
mean,22.261794,3.808897,1.611235,247.850177,104.788352,4.081306,251.74059,174.458391,7.407382,-3.116408,...,9.485313,-1.277398,1.024062,1.003183,1.095608,0.032695,1.025399,1.005389,-0.010199,0.097030
std,19.817416,8.968328,14.075478,537.060587,9.127199,21.639895,626.72270,459.579877,26.990237,268.045701,...,18.805072,5.838412,0.360804,0.573478,0.493540,0.059549,0.205035,0.183969,0.056144,0.296029
min,-6.420000,-41.760000,-81.280000,-324.060000,70.540000,-116.990000,-562.63000,-487.990000,-50.920000,-1563.420000,...,-92.630000,-37.490000,0.250000,-1.920000,0.320000,0.000000,0.510000,0.500000,-0.190000,0.000000
25%,10.230000,1.460000,0.200000,15.220000,101.487500,1.137500,6.11500,2.062500,-4.412500,-44.932500,...,5.060000,-1.780000,0.870000,0.900000,0.900000,0.000000,0.930000,0.930000,-0.040000,0.000000
50%,16.260000,4.120000,2.995000,82.940000,104.300000,6.900000,75.58500,52.900000,3.670000,-2.885000,...,10.550000,-0.860000,0.980000,1.000000,1.000000,0.010000,1.010000,0.990000,-0.010000,0.000000
75%,27.190000,7.490000,6.470000,252.600000,108.100000,13.012500,263.24500,186.040000,13.712500,37.650000,...,17.012500,-0.080000,1.100000,1.100000,1.130000,0.030000,1.090000,1.050000,0.020000,0.000000
max,100.000000,28.050000,38.010000,3531.470000,138.990000,49.480000,4312.26000,3147.480000,166.090000,1175.330000,...,58.530000,21.560000,3.180000,3.930000,4.040000,0.300000,1.940000,1.860000,0.160000,1.000000


In [43]:
model = LogisticRegression(random_state = 42)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print(matrix)

Accuracy: 0.9686758424299953
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
[[2041   23]
 [  43    0]]


In [44]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.9798616761594793
Precision: 0.5
Recall: 0.010101010101010102
F1 Score: 0.019801980198019806


임의의 이상치 처리로 성능이 향상되기 보다는 오히려 떨어짐

### 3-2.

In [45]:
import numpy as np
from scipy.stats import mstats
import statsmodels.api as sm

# 중앙값과 절대 중앙값 편차 (MAD) 계산 함수
def mad_based_outliers(df, threshold=3):        # 임계값은 3으로 설정
    median = np.median(df)
    mad = sm.robust.scale.mad(df)
    mad_scaled = np.abs(df - median) / mad if mad != 0 else 0
    return mad_scaled > threshold

# 윈저라이징 조건 설정 함수
def apply_winsorization_based_on_outliers(df, threshold=3):
    winsorization_info = {}  # 각 컬럼별 윈저라이징 방식을 저장할 딕셔너리

    for column in df.columns:
        # MAD 기반 이상치 탐지
        outliers_mask = mad_based_outliers(df[column], threshold)
        outliers_high = np.sum(outliers_mask & (df[column] > np.median(df[column])))
        outliers_low = np.sum(outliers_mask & (df[column] < np.median(df[column])))

        # 조건에 따라 윈저라이징 방식 결정
        if outliers_high > 0 and outliers_low == 0:  # 상위 이상치만 있을 때
            df[column] = mstats.winsorize(df[column], limits=(0, 0.05))  # 상위 5%만 윈저라이징
            winsorization_info[column] = "상위 2% 윈저라이징"
        elif outliers_low > 0 and outliers_high == 0:  # 하위 이상치만 있을 때
            df[column] = mstats.winsorize(df[column], limits=(0.05, 0))  # 하위 5%만 윈저라이징
            winsorization_info[column] = "하위 2% 윈저라이징"
        elif outliers_high > 0 and outliers_low > 0:  # 상하위 이상치가 모두 있을 때
            df[column] = mstats.winsorize(df[column], limits=(0.25, 0.25))  # 상하위 2.5%씩 윈저라이징
            winsorization_info[column] = "상하위 1% 윈저라이징"
    return winsorization_info

In [46]:
winsorization_info = apply_winsorization_based_on_outliers(X_train)

In [47]:
model = LogisticRegression(random_state = 42)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print(matrix)

Accuracy: 0.9520645467489322
Precision: 0.016666666666666666
Recall: 0.023255813953488372
F1 Score: 0.01941747572815534
[[2005   59]
 [  42    1]]


In [48]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.9798616761594793
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


## 스케일링

### 1. train set에 Standard 적용

In [49]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
sd_scale = StandardScaler()
# sd_scale.fit(X_train)
X_train_scaled = sd_scale.fit_transform(X_train)

In [50]:
model = LogisticRegression(random_state = 42)
model.fit(X_train_scaled, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matrix= confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print(matrix)

Accuracy: 0.4266729947793071
Precision: 0.02526487367563162
Recall: 0.7209302325581395
F1 Score: 0.048818897637795275
[[ 868 1196]
 [  12   31]]


In [51]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.45748576078112285
Precision: 0.02932551319648094
Recall: 0.8080808080808081
F1 Score: 0.05659709939865583


### 2. train set에 MinMax 적용

In [52]:
mm_scale = MinMaxScaler()
# sd_scale.fit(X_train)
X_train_scaled = mm_scale.fit_transform(X_train)

In [53]:
model = LogisticRegression(random_state = 42)
model.fit(X_train_scaled, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matrix= confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print(matrix)

Accuracy: 0.13194114855244424
Precision: 0.021948608137044967
Recall: 0.9534883720930233
F1 Score: 0.042909471480900054
[[ 237 1827]
 [   2   41]]


In [54]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.08848657445077299
Precision: 0.021197552447552448
Recall: 0.9797979797979798
F1 Score: 0.04149732620320856


### 3. train set에 Standard 적용, test set 역시 적용

In [55]:
X_train_scaled = sd_scale.fit_transform(X_train)
X_test_scaled = sd_scale.transform(X_test)

In [56]:
model = LogisticRegression(random_state = 42)
model.fit(X_train_scaled, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test_scaled)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matrix= confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print(matrix)

Accuracy: 0.9387755102040817
Precision: 0.061224489795918366
Recall: 0.13953488372093023
F1 Score: 0.08510638297872342
[[1972   92]
 [  37    6]]


In [57]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.45748576078112285
Precision: 0.02932551319648094
Recall: 0.8080808080808081
F1 Score: 0.05659709939865583


### 4. train set에 MinMax 적용, test set 역시 적용

In [58]:
X_train_scaled = mm_scale.fit_transform(X_train)
X_test_scaled = mm_scale.transform(X_test)

In [59]:
model = LogisticRegression(random_state = 42)
model.fit(X_train_scaled, y_train)

# 테스트 데이터에 대한 예측
y_pred = model.predict(X_test_scaled)

# 다양한 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matrix= confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print(matrix)

Accuracy: 0.9530137636449929
Precision: 0.04838709677419355
Recall: 0.06976744186046512
F1 Score: 0.05714285714285714
[[2005   59]
 [  40    3]]


In [60]:
y_train_pred = model.predict(X_train)
accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.08848657445077299
Precision: 0.021197552447552448
Recall: 0.9797979797979798
F1 Score: 0.04149732620320856


minmax, standard 모두 성능이 좋지는 않지만 standard가 좀 더 좋음